In [ ]:
query = """

select * from Tmp_LLamadas_Efectiva_Negocios
where Fecha_Llamada='2026-08-28'
and Nombre_Campana in ('2026-08 EFECTIVA_NEGOCIOS SEG','2026-08 EFECTIVA_NEGOCIOS CET')
    """
df_ultimo_negocios=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [3]:
df_ultimo_negocios.count()

3319

## actualizar retiro telef 

In [1]:
import sys 
sys.path.append('C:/Users/Data/Documents/lazo_fernando/target_script_01/funciones')
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
from funciones import *
from funciones_spark import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.4.0.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

    

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)



filename='Consulta_de_Campañas_202608_V3_SS_02.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,',',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS_01.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,',',True,ruta_alfin)
df_validar=df_validar_01.unionByName(df_validar_02)


In [28]:
print(df_validar_02.columns)

['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TASA_MIN_DESCUENTO', 'TIPO']


In [29]:
filename='validar_campana_alfin.csv'
df_bloqueo=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

In [5]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where fecha_envio>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)


In [ ]:
print(df_bloqueo.count())

4533


In [34]:
print(df_bloqueo.join(df_validar,['DNI'],'inner').count())


3100


In [35]:
df_bloqueo=df_bloqueo.join(df_validar,['DNI'],'inner')

In [2]:

import sys 
sys.path.append('C:/Users/Data/Documents/lazo_fernando/target_script_01/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
filename='Out_JV_20260828_Priorizados.xlsx'
filePath = os.path.join(ruta_alfin, filename)
df_formato1 = pd.read_excel(filePath)

In [4]:
df_formato1["NumeroDocumento"] = (
    df_formato1["NumeroDocumento"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(8)
)
df_formato1.head()

,NumeroDocumento,TASA_NUEVA
0,42117487,0.49
1,22424690,0.71
2,09418895,0.61
3,08984549,0.39
4,09872000,0.66


In [8]:
df_formato1 = df_formato1.rename(
    columns={'NumeroDocumento': 'dni_cliente'}
)

In [ ]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where fecha_envio>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)


In [6]:
df_formato1.count()

NumeroDocumento    5497
TASA_NUEVA         5497
dtype: int64

In [11]:
df_formato=df_formato1.merge(
    df_prospectos_correos_alfin,
    on='dni_cliente',
    how='inner'
).drop_duplicates(
    subset=['dni_cliente']
)

In [12]:

ruta_archivo = os.path.join(ruta_alfin, 'sssubir.csv')
df_formato.to_csv(ruta_archivo, sep=';')

In [ ]:
filename='Out_JV_20260828_Priorizados.xlsx'
filePath = os.path.join(ruta_alfin, filename)
df_formato1 = pd.read_excel(filePath)
filename='BLOQUEO ALFIN ADICIONAL 27-08.xlsx'
filePath = os.path.join(ruta_alfin, filename)
df_formato = pd.read_excel(filePath)

Index(['dni', 'cliente', 'celular', 'codigo', 'agencia', 'monto'], dtype='str')

In [ ]:
df_formato1 = df_formato1[['dni', 'cliente', 'celular', 'agencia', 'monto']]
df_formato  = df_formato[['DNI', 'CLIENTE', 'CELULAR', 'AGENCIA', 'MONTO']]

df_formato1.columns = df_formato1.columns.str.upper()

# Unir ambos DataFrames
df_final = pd.concat(
    [df_formato, df_formato1],
    ignore_index=True
)


In [16]:
df_seg1=df_seg1[
    ~df_seg1['dni_cliente'].isin(dni_retiro)&
    ~df_seg1['celular'].isin(cel_retiro)&
    ~df_seg1['dni_cliente'].isin(set_tipi)&
    ~df_seg1['dni_cliente'].isin(dni_desembolso)
    ].copy()
df_seg1.shape


(4038, 23)

In [4]:
df_01=df_ultimo_negocios.toPandas()

In [5]:

ruta_archivo = os.path.join(ruta_alfin, 'negocios_hoy.csv')
df_01.to_csv(ruta_archivo, sep=';')

In [36]:
df_bloqueo_pd=df_bloqueo.toPandas()

c:\Users\Data\Documents\lazo_fernando\target_script_01\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\Data\Documents\lazo_fernando\target_script_01\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


,DNI,CLIENTE,CELULAR,AGENCIA,MONTO


In [21]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where estado='ENVIADO'
    and fecha_envio>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)


In [ ]:
df_final_notiene=df_final[
    ~df_final['DNI'].isin(df_prospectos_correos_alfin['dni_cliente'].unique())
    ].copy()
df_final_notiene.shape


In [24]:
df_final_tiene=df_final[
    df_final['DNI'].isin(df_prospectos_correos_alfin['dni_cliente'].unique())
    ].copy()
df_final_tiene.shape

(0, 5)

In [ ]:

c
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)

horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
# df_formato['dni_vendedor']=df_formato['ejecutivo_target']
# df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'

In [15]:
df_formato = df_formato.rename(columns={
    'Agencia_comercial': 'agencia_atencion'
})

In [12]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [13]:
df_formato.shape

(1538, 23)

In [6]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [17]:
df_agencia['agencia_correo'] = (
    df_agencia['agencia_correo']
    .replace("CAÃ‘ETE", "CAÑETE")
)

In [26]:
df_formato = df_formato[
    ~df_formato['agencia_atencion'].isin(['CAÑETE','CAÑETE'])
].copy()

In [31]:
df_formato=df_formato.merge(df_agencia[['agencia_atencion','cod_agencia']],on='agencia_atencion',how='inner')

In [32]:
df_formato.head()

,dni_cliente,PROPENSION_IC,frescura,PHONE_NUMBER,cruce,lote,color_final,agencia_atencion,NOMBRES,monto_solicitado,...,supervisor,canal_campo,codigo_ejecutivo_id,ejecutivo_target,cdv_alfin_banco,hora_visita,telefono_cliente,operador,tipo_gestion,cod_agencia
0,80517046,1,0,939312859,CET,NO CLIENTE,VERDE OSCURO,CHICLAYO BALTA,MERY DORA,8300,...,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,10:15:00,939312859,TARGET,Derivacion,734281 - CHICLAYO BALTA
1,00252748,1,0,972629681,CET,NO CLIENTE,VERDE OSCURO,SULLANA,GLADYS MARIBEL,11200,...,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,14:00:00,972629681,TARGET,Derivacion,734270 - SULLANA
2,47835526,1,0,917252906,CET,NO CLIENTE,AMARILLO OSCURO,AREQUIPA CAYMA,MERY,10000,...,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,10:45:00,917252906,TARGET,Derivacion,738397 - AREQ CAYMA
3,45968472,1,0,965413477,CET,NO CLIENTE,VERDE OSCURO,MOSHOQUEQUE,SONIA MERCEDES,17300,...,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,17:45:00,965413477,TARGET,Derivacion,738360 - MOSHOQUEQUE
4,02438702,1,0,950084673,CET,NO CLIENTE,AMARILLO OSCURO,JULIACA 2,RUBEN ROGELIO,10100,...,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,15:15:00,950084673,TARGET,Derivacion,735986 - JULIACA 2


In [ ]:
# df_formato['agencia_tienda']=df_formato['cod_agencia']


In [30]:
query = f"""
	select *,agencia_correo as agencia_atencion ,agencia_Formulario as cod_agencia from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_atencion']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'CAÑETE'}
{' AREQ PAMPILLA', ' ENMANCIPACION', ' CHIMBOTE', ' MIRAFLORES', ' TRUJ CENTRO', ' LOS OLIVOS', ' PUCALLPA', ' SANTA ANITA', ' CUSCO LA CULTURA', ' PUENTE PIEDRA', ' TRUJ AMERICA', ' SULLANA', 'CAÃ‘ETE', ' CAÃ‘ETE', ' CHINCHA', ' SAN MIGUEL', ' PC HUANCAYO', ' MOSHOQUEQUE', ' TUMBES', ' VILLA MARIA 2', ' HUANUCO', ' SAN JUAN DE LURIG', ' CHICLAYO BALTA', ' VENTANILLA', ' PISCO', ' HUARAL', ' TARAPOTO', ' IQUITOS', ' AREQ CAYMA', ' ICA', ' PC HUARAZ', ' CAJAMARCA', ' CASTILLA', ' JULIACA 2', ' SAN MARTIN', ' COMAS', ' JESUS MARIA', ' HUACHO', ' SAN JUAN DE MIRAFLORES', ' VILLA EL SALVADOR 2', ' ATE VITARTE', ' PC TACNA'}


In [26]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'AREQUIPA CAYMA', 'TRUJILLO CENTRO', 'TACNA', 'HUARAZ', 'SAN JUAN DE LURIGANCHO', 'TRUJILLO AMERICA', 'AREQUIPA PAMPILLA', 'HUANCAYO', 'EMANCIPACION'}
{'AREQ PAMPILLA', 'PC HUARAZ', 'ENMANCIPACION', 'AREQ CAYMA', 'TRUJ CENTRO', 'SAN JUAN DE LURIG', 'TRUJ AMERICA', 'PC HUANCAYO', 'PC TACNA'}


#### validar el codigo de agencia 

In [11]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'730879 - PAITA', '734299 - CUSCO LA CULTURA', '739580 - ICA', '738364 - TRUJ AMERICA', '739470 - HUARAL', '734280 - PC HUANCAYO', '734270 - SULLANA', '732243 - CAÑETE', '738363 - CAJAMARCA', '738381 - ENMANCIPACION', '739629 - ATE VITARTE', '734285 - PC HUARAZ', '738360 - MOSHOQUEQUE'}
set()


In [24]:
df_formato['agencia_tienda'] = (
    df_formato['agencia_atencion']
    .replace("CAÃ‘ETE", "CAÑETE")
)

In [ ]:
df_formato = df_formato[
    ~df_formato['agencia_tienda'].isin(['CASTILLA', 'AREQUIPA PAMPILLA', '734281 -  CHICLAYO BALTA', 'JESUS MARIA'])
].copy()

In [19]:
print(df_agencia["agencia_correo"].drop_duplicates().tolist())

['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']


In [ ]:
['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']MARIA 

In [19]:

equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [34]:
print(df_correo["agencia_atencion"].drop_duplicates().tolist())


['HUANUCO', 'HUANCAYO', 'SAN MIGUEL', 'CUSCO LA CULTURA', 'TACNA', 'TRUJILLO CENTRO', 'TARAPOTO', 'EMANCIPACION', 'CHIMBOTE', 'MIRAFLORES', 'TRUJILLO AMERICA', 'HUACHO', 'COMAS', 'CHICLAYO BALTA', 'SAN JUAN DE LURIGANCHO', 'CASTILLA', 'AREQUIPA PAMPILLA', 'SULLANA', 'VENTANILLA', 'MOSHOQUEQUE', 'JESUS MARIA', 'PUCALLPA', 'ATE VITARTE', 'LOS OLIVOS', 'VILLA MARIA 2', 'SANTA ANITA', 'CAJAMARCA', 'AREQUIPA CAYMA', 'PISCO', 'HUARAZ', 'SAN JUAN DE MIRAFLORES', 'CHINCHA', 'HUARAL', 'ICA', 'SAN MARTIN', 'JULIACA 2', 'VILLA EL SALVADOR 2', 'TUMBES', 'CAÑETE', 'PUENTE PIEDRA', nan, 'TE']


In [ ]:

df_correo[df_correo['dni_cliente']=='09704310'].head()

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
2190,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09704310,SALVADOR ALBERTO CHOQUE ALARCON,NaN,18000,930162239,MIRAFLORES,2026-07-15,13:30:00,MANUAL


In [12]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=',')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)




dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())


C:\Users\Data\AppData\Local\Temp\ipykernel_5348\3600096180.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [11]:
df_retiros.columns

Index(['dni_cliente', 'celular'], dtype='object')

In [4]:
df_retiros[df_retiros['dni_cliente']=='41999928'].head()

,dni_cliente,celular


In [32]:
filename='alfin_ult_2.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_qui = pd.read_csv(ruta_archivo,sep=';')

In [35]:
df_qui.rename(columns={'DNI':'dni_cliente'},inplace=True)

In [37]:
df_ref=df_qui[['dni_cliente','COLOR_FINAL']].copy()

In [46]:
df_formato['dni_cliente'] = (
    df_formato['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

In [47]:
df_formato_1=df_formato.merge(df_ref,on='dni_cliente',how='inner')

In [48]:

df_formato_1.count()

dni_cliente            2849
nombre_cliente         2849
celular                2849
cod_agencia            2849
agencia_atencion       2849
fecha_visita           2849
monto_solicitado       2849
color_1                   0
supervisor             2849
canal_campo            2849
codigo_ejecutivo_id    2849
ejecutivo_target       2849
cdv_alfin_banco        2849
hora_visita            2849
telefono_cliente       2849
dni_vendedor           2849
agencia_tienda         2849
operador               2849
tipo_gestion           2849
COLOR_FINAL            2849
dtype: int64

In [22]:
filename='quitar_.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_haber = pd.read_excel(filePath)


In [9]:
df_prospectos_envio.shape


(21149, 21)

In [ ]:
df_seg=df_seg[
    ~df_seg['dni_cliente'].isin(dni_retiro)&
    ~df_seg['celular'].isin(cel_retiro)&
    ~df_seg['dni_cliente'].isin(dni_desembolso)
    ].copy()
df_seg.shape


(1897, 24)

In [4]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where fecha_envio>='2026-08-24'
"""
df_prospectos_envio = pd.read_sql(query, engine_mysql)
df_prospectos_envio=df_prospectos_envio.drop_duplicates(['dni_cliente'])
df_prospectos_envio.shape

(3019, 21)

In [6]:
ruta_csv

'C:\\Users\\DATA\\Documents\\datos\\05_subir_csv'

In [9]:

ruta_archivo = os.path.join(ruta_alfin, 'suvir_vici_alfin.csv')
df.to_csv(ruta_archivo, sep=';')

In [24]:
df_haber.columns

Index(['vendor_lead_code', 'phone_number'], dtype='object')

In [23]:
df_haber.shape

(13888, 2)

In [13]:
server_sql = server_zeus
db_sql = "THOTH"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	SELECT distinct Dni,Descripcion_ FROM THOTH.dbo.Tmp_LLamadas_Alfin 
    where Descripcion_ in(
    'TELEFONO FUERA DE SERVICIO / NO EXISTE',
       'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
       'SOLICITÓ NO SER CONTACTADO',
       'FUERA DE SERVICIO',
       'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
       'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES'
    )
"""
df_tipis = pd.read_sql(query, engine)
set_tipi = set(
    df_tipis['Dni']
    .dropna()
    .drop_duplicates()
)



## aca

In [3]:
query = f"""
	SELECT dni_cliente,'enviado' as estado17 FROM Alice.prospectos_envio_alfin 
    where estado='procesado'
    and fecha_envio>='2026-08-01'
    and fecha_envio<'2026-08-28'
"""
df_prospectos_envio_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where estado='ENVIADO'
    and fecha_envio>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)


In [ ]:
df_hoy

In [4]:
df_prospectos_correos_alfin["q_envio"] = (
    df_prospectos_correos_alfin.groupby("dni_cliente")["dni_cliente"]
    .transform("count")
)
df_prospectos_correos_alfin["cantidad_repeticiones"] = (
    df_prospectos_correos_alfin
    .groupby("dni_cliente")["dni_cliente"]
    .transform("count")
)


df_prospectos_correos_alfin["fecha_envio"] = pd.to_datetime(
    df_prospectos_correos_alfin["fecha_envio"],
    errors="coerce"
)
df_prospectos_correos_alfin = df_prospectos_correos_alfin.sort_values(
    "fecha_envio",
    ascending=False
)
df_prospectos_correos_alfin = (
    df_prospectos_correos_alfin
    .drop_duplicates(
        subset="dni_cliente",
        keep="first"
    )
    .reset_index(drop=True)
)


In [5]:
df_seg=df_prospectos_correos_alfin.merge(df_prospectos_envio_alfin[['dni_cliente']],on='dni_cliente',how='inner').drop_duplicates(subset=['dni_cliente'])

In [7]:
df_seg.shape

(24620, 23)

In [6]:
df_prospectos_correos_alfin.shape

(28754, 23)

In [20]:
import pandas as pd

# Convertir a fecha
df_prospectos_correos_alfin["fecha_envio"] = pd.to_datetime(
    df_prospectos_correos_alfin["fecha_envio"],
    errors="coerce"
)

# Cantidad de días desde fecha_envio hasta hoy
df_prospectos_correos_alfin["q_dias"] = (
    pd.Timestamp.today().normalize()
    - df_prospectos_correos_alfin["fecha_envio"].dt.normalize()
).dt.days

In [21]:
df_prospectos_correos_alfin["fecha_envio"] = pd.to_datetime(
    df_prospectos_correos_alfin["fecha_envio"],
    errors="coerce"
).dt.normalize()

In [14]:
df_estado = (
    df_prospectos_correos_alfin
    .groupby('fecha_envio')
    .size()
    .reset_index(name='cantidad')
)

df_estado

,fecha_envio,cantidad
0,2026-08-06,846


In [10]:
df_seg["fecha_envio"] = pd.to_datetime(
    df_seg["fecha_envio"],
    errors="coerce"
)

condicion = (
    (
        (df_seg["q_envio"] > 1)
        &
        (df_seg["fecha_envio"].dt.normalize().isin([
            "2026-08-27"
        ]))
    )
    |
    (
        (df_seg["q_envio"] == 1)
        &
        (df_seg["fecha_envio"].dt.normalize() >= "2026-08-25")
    )
)

df_seg1 = df_seg[condicion].copy()

C:\Users\Data\AppData\Local\Temp\ipykernel_5348\1328488329.py:10: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  (df_seg["fecha_envio"].dt.normalize().isin([


In [11]:
df_seg1.shape

(4077, 23)

In [9]:
df_seg1.shape


(3205, 23)

In [7]:
df_seg.shape

(2374, 22)

In [76]:
df.shape

(7133, 21)

In [13]:
df_seg1=df_seg1[
    ~df_seg1['dni_cliente'].isin(dni_retiro)&
    ~df_seg1['celular'].isin(cel_retiro)&
    ~df_seg1['dni_cliente'].isin(dni_desembolso)&
    ~df_seg1['dni_cliente'].isin(set_tipi)
    ].copy()
df_seg1.shape




(3096, 23)

In [ ]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where fecha_envio>='2026-08-22'
"""
df_prospectos_envio = pd.read_sql(query, engine_mysql)

df_prospectos_envio=df_prospectos_envio.drop_duplicates('dni_cliente')

ruta_archivo = os.path.join(ruta_csv, 'subir_correo.csv')
df_prospectos_envio.to_csv(ruta_archivo, sep=';')


In [ ]:
df_prospectos_envio

In [51]:
df_prospectos_envio=df_prospectos_envio.merge(df_seg[['dni_cliente']],on='dni_cliente',how='inner')
df_prospectos_envio = df_prospectos_envio.drop_duplicates(subset="dni_cliente")
df_prospectos_envio.shape


(2626, 16)

In [13]:
df_formato.columns

Index(['dni_cliente', 'nombre_cliente', 'celular', 'cod_agencia',
       'agencia_atencion', 'fecha_visita', 'monto_solicitado', 'color_1',
       'supervisor', 'canal_campo', 'codigo_ejecutivo_id', 'ejecutivo_target',
       'cdv_alfin_banco', 'hora_visita', 'telefono_cliente', 'dni_vendedor',
       'agencia_tienda', 'operador', 'tipo_gestion'],
      dtype='object')

In [14]:
df_formato=df_formato.rename(columns={'color_1':'color'})

In [67]:
df_correo = df_correo.drop_duplicates(subset=['dni_cliente'])

df_formulario = df_formulario.drop_duplicates(subset=['dni_cliente'])

In [70]:
print(df_correo.shape)
print(df_formulario.shape)

(67, 14)
(67, 9)


In [41]:
df_formato.rename(columns={'color_final':'color'},inplace=True)


In [ ]:
df_formato['dni_vendedor']='BOT'
df_formato['agencia_tienda']=df_formato['agencia_atencion']


In [32]:
df_correo=df_seg1[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita','color']] .copy()
df_correo['tipo_carga']='MANUAL'

df_formulario=df_seg_formulario[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,color,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,09260966,LUISA FILONILA QUISPE,15900.0,925596220,VILLA MARIA 2,2026-08-31,10:15:00,AMARILLO OSCURO,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,10754569,ANGELICA VASQUEZ TICLIA,11000.0,918775801,VILLA EL SALVADOR 2,2026-08-29,18:45:00,AMARILLO OSCURO,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,05593302,PALMIRA GUERRA TENAZOA,964851592,733824 - TARAPOTO,2026-08-31,20000,Derivacion
1,BOT,TARGET,03492826,HEBERTH RICARDO T<ALLEDO DE LAMA,902730843,737490 - CASTILLA,2026-08-29,8000,Derivacion


In [21]:
import pandas as pd
import numpy as np

fechas = pd.to_datetime([
    "2026-08-29",
    "2026-08-31",
])

df_seg1["fecha_visita"] = np.random.choice(
    fechas,
    size=len(df_seg1)
)


In [22]:
# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_seg1))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_seg1))

# Crear la columna
df_seg1["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

In [34]:
df_formato.head(3)

,dni_cliente,PROPENSION_IC,frescura,PHONE_NUMBER,cruce,lote,color_final,agencia_atencion,NOMBRES,monto_solicitado,...,canal_campo,codigo_ejecutivo_id,ejecutivo_target,cdv_alfin_banco,hora_visita,telefono_cliente,operador,tipo_gestion,cod_agencia,fecha_visita
0,80517046,1,0,939312859,CET,NO CLIENTE,VERDE OSCURO,CHICLAYO BALTA,MERY DORA,8300,...,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,10:15:00,939312859,TARGET,Derivacion,734281 - CHICLAYO BALTA,2026-08-29
1,00252748,1,0,972629681,CET,NO CLIENTE,VERDE OSCURO,SULLANA,GLADYS MARIBEL,11200,...,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,14:00:00,972629681,TARGET,Derivacion,734270 - SULLANA,2026-08-25
2,47835526,1,0,917252906,CET,NO CLIENTE,AMARILLO OSCURO,AREQUIPA CAYMA,MERY,10000,...,CALL CENTER / TARGET OUTSOURCING,00000001,BOT,ROSA HONOR,10:45:00,917252906,TARGET,Derivacion,738397 - AREQ CAYMA,2026-08-27


In [ ]:
90# maquina 10
100 -120




In [23]:
df=df_seg1[["fecha_visita",'dni_cliente']].copy()

In [24]:
query = f"""
	SELECT * FROM Alice.prospectos_envio_alfin 
    where fecha_creacion>='2026-08-01'
"""
df_seg_formulario = pd.read_sql(query, engine_mysql)

In [25]:
df_seg_formulario = df_seg_formulario.drop(
    columns=["fecha_visita"]
)

In [26]:
df_seg_formulario=df_seg_formulario.merge(df,on='dni_cliente',how='inner')
df_seg_formulario=df_seg_formulario.drop_duplicates(subset=['dni_cliente'])

In [92]:
df_seg_formulario.head(2)

,id,hash_duplicado,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,monto_solicitado,tipo_gestion,estado,codigo_http_ms,respuesta_ms,fecha_creacion,fecha_envio,fecha_visita
0,222560,None,BOT,TARGET,80644018,LUIS ALBERTO ALCANTARA CAPU+æAY\t,940755112,734281 - CHICLAYO BALTA,10000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-08-02 06:00:00,2026-08-02 12:02:02,2026-08-24
1,222577,None,BOT,TARGET,80234657,MARIA TEMPORA TIMANA LACHIRA,956619138,732243 - CAÃ‘ETE,11000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-08-01 07:52:43,2026-08-01 20:18:20,2026-08-22


In [46]:
df_seg_formulario.shape

(823, 16)

In [27]:
df_seg1=df_seg1.merge(df_seg_formulario[['dni_cliente']],on='dni_cliente',how='inner')


In [28]:
df_seg_formulario=df_seg_formulario.drop_duplicates(subset=['dni_cliente'])
df_seg1=df_seg1.drop_duplicates(subset=['dni_cliente'])


In [29]:
df_seg_formulario.shape

(4038, 16)

In [30]:
df_seg1.shape


(4038, 23)

In [33]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

4038

In [22]:
df_correo_pendiente=df_correo.copy()
df_formulario_pendiente=df_formulario.copy()

In [44]:
df_formulario_pendiente[df_formulario_pendiente['dni_cliente']=='80248715'].head()

,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion


In [45]:
df_formulario[df_formulario['dni_cliente']=='80248715'].head()


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
15,BOT,TARGET,80248715,YOLANDA MARIA OLIVERO AGUILAR,991554388,737166 - MIRAFLORES,2026-08-02,4700,Derivacion


In [ ]:
df_prospectos_envio=df_prospectos_envio.drop_duplicates('dni_cliente')


In [75]:

ruta_archivo = os.path.join(ruta_csv, 'subir_vici.csv')
df.to_csv(ruta_archivo, sep=';')


In [9]:
df_seg.shape

(2374, 22)

In [46]:
df_formulario_actualizado = pd.read_csv(ruta_archivo,sep=';')

In [ ]:
df_formulario_actualizado['fecha_visita']=df_formulario_actualizado['fecha_visita']

In [51]:
df_formulario_actualizado["fecha_visita"] = pd.to_datetime(
    df_formulario_actualizado["fecha_visita"],
    errors="coerce"
)

In [ ]:
df_formulario_actualizado

In [48]:
df_formulario_actualizado['dni_cliente'] = (
    df_formulario_actualizado['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)      # deja solo números
    .replace('', pd.NA)                      # vacío -> NA
    .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)

In [3]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where fecha_envio>='2026-08-26'

"""
df = pd.read_sql(query, engine_mysql)


In [7]:
df = df.drop_duplicates(subset=["dni_cliente"])

In [8]:

df.shape

(6482, 21)

In [73]:
df=df.drop_duplicates(['dni_cliente'])


(5053, 21)

In [57]:
df_seg_formulario =df_prospectos_envio_alfin.merge(df_seg[['dni_cliente']],on='dni_cliente',how='inner')

In [58]:
df_seg_formulario=df_seg_formulario.drop_duplicates('dni_cliente')


In [59]:
df_seg_formulario.columns

Index(['id', 'hash_duplicado', 'dni_vendedor', 'operador', 'dni_cliente',
       'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita',
       'monto_solicitado', 'tipo_gestion', 'estado', 'codigo_http_ms',
       'respuesta_ms', 'fecha_creacion', 'fecha_envio'],
      dtype='object')

In [ ]:
df_correo = df_correo.drop_duplicates(subset=['dni_cliente'])

df_formulario = df_formulario.drop_duplicates(subset=['dni_cliente'])

In [69]:
dni_s=set(df['dni_cliente'].unique())

In [99]:
dni_s=set(df_no_Cargar['DNI'].unique())


In [100]:
df=df[~df['dni_cliente'].isin(dni_s)].copy()


In [ ]:
# dni_s=set(df['dni_cliente'].unique())
df_correo=df_correo[~df_correo['dni_cliente'].isin(dni_s)].copy()
df_formulario=df_formulario[~df_formulario['dni_cliente'].isin(dni_s)].copy()

In [101]:
df.shape

(6599, 21)

In [90]:
filename='NO CARGAR.xlsx'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_no_Cargar = pd.read_excel(ruta_archivo)

In [94]:
df_no_Cargar['DNI'] = (
    df_no_Cargar['DNI']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.replace(r'\D', '', regex=True)
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)

In [95]:
df_no_Cargar.head(2)

,DNI
0,09891347
1,43173825


In [ ]:
df_no_Cargar['DNI'] = (
    df_no_Cargar['DNI']
    .astype(str)
    
    .replace('', pd.NA)                      # vacío -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros

)

In [80]:
df_no_Cargar['DNI'] = (
    df_no_Cargar['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)      # deja solo números
    .replace('', pd.NA)                      # vacío -> NA
    .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)